# Module 24 — CPython Internals

## Exercise 24.1 — Fifteen disassemblies, fifteen questions

Several of these settle arguments from earlier modules. Predict the ANSWER to
each question before running.
Run:  python ex01_dis.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. `dis`: reading bytecode

In [ ]:
import dis

def add(a, b):
    return a + b

dis.dis(add)

```text
  2   RESUME               0
  3   LOAD_FAST            0 (a)
      LOAD_FAST            1 (b)
      BINARY_OP            0 (+)
      RETURN_VALUE
```


A **stack machine**: push `a`, push `b`, pop two and push their sum, return the
top. The whole instruction set is about 120 opcodes and you need to recognise
perhaps fifteen.

| Opcode | Means |
|---|---|
| `LOAD_FAST` / `STORE_FAST` | A **local** — an array index. Fast. |
| `LOAD_GLOBAL` | A **global** — a dict lookup, then builtins. Slower. |
| `LOAD_CONST` | A constant baked into the code object |
| `LOAD_ATTR` / `STORE_ATTR` | Attribute access — the Module 08 ladder |
| `LOAD_METHOD` / `CALL` | A method call, avoiding a bound-method allocation |
| `BINARY_OP` | Any binary operator, with an operand saying which |
| `COMPARE_OP` | `<`, `==`, etc. |
| `POP_JUMP_IF_FALSE` | A branch |
| `FOR_ITER` | The iterator protocol, in one instruction |
| `MAKE_FUNCTION` | A `def` or `lambda` executing |
| `BUILD_LIST` / `LIST_APPEND` | Comprehension machinery |

**`dis` settles arguments.** Any time two people disagree about what Python does,
disassemble it:

In [ ]:
dis.dis("x = 1 + 2")              # LOAD_CONST 3 -- folded (Module 01)
dis.dis("a += 1")                 # LOAD, ADD, STORE -- three ops (Module 21)
dis.dis("[x for x in y]")         # its own code object (Module 04)
dis.dis("f'{x}'")                 # FORMAT_VALUE, not string concatenation

---

## Concept 2. Code objects and frames

In [ ]:
def outer(a, b=2, *args, **kwargs):
    x = a + b
    def inner(): return x
    return inner

c = outer.__code__
c.co_varnames        # ('a', 'b', 'args', 'kwargs', 'x', 'inner')
c.co_consts          # constants, INCLUDING inner's code object
c.co_names           # global and attribute names referenced
c.co_freevars        # names captured FROM an enclosing scope
c.co_cellvars        # ('x',) -- names captured BY an inner function
c.co_argcount, c.co_flags, c.co_stacksize

`co_cellvars` is Module 04's closure cell, made visible. `x` is in it precisely
because `inner` reads it — that is what turns a local into a cell.

A **frame** is created per call and holds the locals array, the value stack, and
a pointer back to the caller. That chain of pointers is what a traceback walks
(Module 01), and what keeps every local alive while an exception is stored
(Module 02).

In [ ]:
import sys
frame = sys._getframe()
frame.f_locals, frame.f_back, frame.f_code.co_name

---

## Concept 5. The specialising adaptive interpreter (3.11+)

This is the biggest change to CPython's execution model in a decade, and it is
why old performance folklore is unreliable.

The interpreter **watches** which types actually flow through each bytecode and
rewrites the instruction in place into a specialised form:

```text
BINARY_OP  (generic: check types, dispatch, maybe call __add__)
    ↓ after a few iterations where both operands are always ints
BINARY_OP_ADD_INT  (a direct integer add, with a guard that falls back)
```


Consequences worth carrying:

- **Monomorphic code is faster than polymorphic code.** A loop where a variable
  is always an `int` specialises; one where it is sometimes an `int` and
  sometimes a `str` cannot, and de-optimises.
- **Benchmarks need a warm-up.** The first few iterations run unspecialised.
- **Micro-benchmark folklore from before 3.11 is often wrong now.** Re-measure
  rather than repeating what you read.

3.12 added a JIT-adjacent "Tier 2" IR, and 3.13 shipped an experimental
copy-and-patch JIT. The direction is clear: **the interpreter is getting
smarter, so measure on your actual version** (Module 23).

---

## Concept 6. Reference counting and the cycle collector, one level down

Module 02 covered the behaviour. The mechanism:

- Every object begins with `ob_refcnt` and `ob_type`. `Py_INCREF`/`Py_DECREF`
  are macros the interpreter calls constantly.
- At zero, the deallocator runs immediately.
- The **generational** cycle collector tracks container objects in three
  generations. Generation 0 is collected often, and survivors are promoted.
  Thresholds default to `(700, 10, 10)`.

In [ ]:
import gc
gc.get_threshold()      # (700, 10, 10)
gc.get_count()          # objects since the last collection, per generation
gc.freeze()             # move everything to a permanent generation

`gc.freeze()` after startup is a real production technique: it moves
long-lived objects out of the collector's way, which matters enormously for a
pre-fork server, where it also stops the collector from touching (and therefore
copy-on-write un-sharing) every page of the parent's heap. Instagram's
well-known memory reduction came largely from this.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `dis`: reading bytecode
- Section 2: Code objects and frames
- Section 3: Where the memory goes
- Section 4: How an attribute lookup really resolves
- Section 5: The specialising adaptive interpreter (3.11+)
- Section 6: Reference counting and the cycle collector, one level down
- Section 7: Reading CPython's source

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import dis
import sys

---

## `show`

_show_

In [ ]:
def show(label: str, source: str, question: str) -> None:
    print(f"\n{'=' * 70}\n{label}\n  source: {source!r}\n  Q: {question}\n")
    dis.dis(compile(source, "<s>", "exec"))

---

## `part_b`

Inspect a code object directly.

In [ ]:
def part_b() -> None:
    """Inspect a code object directly."""
    def outer(a: int, b: int = 2) -> object:
        x = a + b

        def inner() -> int:
            return x
        return inner

    c = outer.__code__
    print(f"\n{'=' * 70}\nPart B: the code object\n")
    for attr in ("co_name", "co_argcount", "co_varnames", "co_names",
                 "co_freevars", "co_cellvars", "co_stacksize", "co_nlocals"):
        print(f"  {attr:<14} {getattr(c, attr)}")
    inner_code = [k for k in c.co_consts if hasattr(k, "co_name")]
    print(f"\n  co_consts contains inner's code object: {bool(inner_code)}")
    if inner_code:
        print(f"  inner.co_freevars: {inner_code[0].co_freevars}")

    print(
        "\n  Q: co_cellvars on outer and co_freevars on inner name the same\n"
        "     variable. What is that variable, and which Module 04 concept are\n"
        "     you looking at?\n"
        "  Q: remove the inner function and re-run. What happens to\n"
        "     co_cellvars, and why?"
    )

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    print(f"Python {sys.version.split()[0]} -- bytecode differs between "
          f"versions, which is itself the lesson.")
    for label, source, question in CASES:
        show(label, source, question)
    part_b()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.